In [18]:

!pip install -U laya



In [19]:
!pip install -U laya

In [20]:
!laya-evals validate /kaggle/working/train.jsonl
!laya-evals validate /kaggle/working/validation.jsonl

241 examples, 1 question id(s): diagnosis
80 examples, 1 question id(s): diagnosis


In [21]:
import laya

model = laya.load("convaiinnovations/laya-multilingual")
print("Laya 모델 로딩 완료")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Laya 모델 로딩 완료


In [22]:
import pandas as pd
import glob

print(glob.glob('/kaggle/input/**/*.csv', recursive=True))

df = pd.read_csv(
    glob.glob('/kaggle/input/**/splits.csv', recursive=True)[0]
)

print(df.head())
print(df.shape)

['/kaggle/input/datasets/kimseongmin99/aaaaaa/splits.csv', '/kaggle/input/datasets/kimseongmin99/bbbbbb/ai4i2020.csv']
    UDI     label power_band   stratum  split
0  7088  공구 마모 고장      정상 범위  공구 마모 고장  train
1  3530  공구 마모 고장      정상 범위  공구 마모 고장  train
2  2441        정상      정상 범위        정상  train
3  4426        정상      정상 범위        정상  train
4  5575        정상      정상 범위        정상  train
(401, 5)


In [23]:
print(df['split'].value_counts())
print(df['label'].value_counts())


split
train         241
validation     80
test           80
Name: count, dtype: int64
label
열 방출 장애     101
정상          100
동력 고장        80
과부하 고장       78
공구 마모 고장     42
Name: count, dtype: int64


In [24]:
import glob

files = glob.glob('/kaggle/input/**/*.csv', recursive=True)

for f in files:
    print(f)

/kaggle/input/datasets/kimseongmin99/aaaaaa/splits.csv
/kaggle/input/datasets/kimseongmin99/bbbbbb/ai4i2020.csv


In [25]:

import pandas as pd

# 1. CSV 파일 불러오기
splits = pd.read_csv(
    "/kaggle/input/datasets/kimseongmin99/aaaaaa/splits.csv"
)

ai4i = pd.read_csv(
    "/kaggle/input/datasets/kimseongmin99/bbbbbb/ai4i2020.csv"
)

# 2. 필요한 센서 데이터 추출
sensor_cols = [
    "UDI",
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
]

# 3. UDI 기준으로 데이터 결합
merged = splits.merge(
    ai4i[sensor_cols],
    on="UDI",
    how="left",
    validate="one_to_one"
)

# 4. 결과 확인
print("전체 데이터 크기:", merged.shape)
print("누락된 센서값:", merged[sensor_cols[1:]].isnull().sum().sum())
print(merged.head())


전체 데이터 크기: (401, 10)
누락된 센서값: 0
    UDI     label power_band   stratum  split  Air temperature [K]  \
0  7088  공구 마모 고장      정상 범위  공구 마모 고장  train                300.6   
1  3530  공구 마모 고장      정상 범위  공구 마모 고장  train                301.9   
2  2441        정상      정상 범위        정상  train                299.1   
3  4426        정상      정상 범위        정상  train                302.5   
4  5575        정상      정상 범위        정상  train                302.4   

   Process temperature [K]  Rotational speed [rpm]  Torque [Nm]  \
0                    310.3                    1648         30.5   
1                    310.9                    1567         39.0   
2                    308.4                    1426         47.9   
3                    310.2                    1547         36.9   
4                    312.0                    1305         49.1   

   Tool wear [min]  
0              217  
1              214  
2               67  
3              103  
4               16  


In [26]:

import json
import os

# Laya에 전달할 판단 질문
questions = {
    "diagnosis": {
        "type": "choice",
        "instructions": "센서 데이터를 보고 설비의 상태를 분류하세요.",
        "criteria": {
            "정상": "설비가 정상적으로 작동하는 상태",
            "열 방출 장애": "열 방출 관련 고장",
            "동력 고장": "동력 관련 고장",
            "과부하 고장": "기계적 과부하 관련 고장",
            "공구 마모 고장": "공구 마모 관련 고장"
        }
    }
}

# 센서 데이터 → JSON 형식 변환
def make_case(row):
    state = {
        "air_temperature_K": float(row["Air temperature [K]"]),
        "process_temperature_K": float(row["Process temperature [K]"]),
        "rotational_speed_rpm": int(row["Rotational speed [rpm]"]),
        "torque_Nm": float(row["Torque [Nm]"]),
        "tool_wear_min": int(row["Tool wear [min]"])
    }

    return {
        "state": json.dumps(state, ensure_ascii=False),
        "questions": questions,
        "expected": {
            "diagnosis": row["label"]
        }
    }

# train / validation / test 분리 저장
for split_name in ["train", "validation", "test"]:
    subset = merged[merged["split"] == split_name]

    path = f"/kaggle/working/{split_name}.jsonl"

    with open(path, "w", encoding="utf-8") as f:
        for _, row in subset.iterrows():
            case = make_case(row)
            f.write(json.dumps(case, ensure_ascii=False) + "\n")

    print(f"{split_name}: {len(subset)}개 저장 완료")

# 학습 데이터 예시 출력
print("\n학습 데이터 예시:")
print(json.dumps(
    make_case(merged.iloc[0]),
    ensure_ascii=False,
    indent=2
))


train: 241개 저장 완료
validation: 80개 저장 완료
test: 80개 저장 완료

학습 데이터 예시:
{
  "state": "{\"air_temperature_K\": 300.6, \"process_temperature_K\": 310.3, \"rotational_speed_rpm\": 1648, \"torque_Nm\": 30.5, \"tool_wear_min\": 217}",
  "questions": {
    "diagnosis": {
      "type": "choice",
      "instructions": "센서 데이터를 보고 설비의 상태를 분류하세요.",
      "criteria": {
        "정상": "설비가 정상적으로 작동하는 상태",
        "열 방출 장애": "열 방출 관련 고장",
        "동력 고장": "동력 관련 고장",
        "과부하 고장": "기계적 과부하 관련 고장",
        "공구 마모 고장": "공구 마모 관련 고장"
      }
    }
  },
  "expected": {
    "diagnosis": "공구 마모 고장"
  }
}


In [27]:

# 기존 학습 데이터
base_train = merged[merged["split"] == "train"].copy()

# 기존 401개와 겹치지 않는 데이터만 추출
remaining = ai4i[
    ~ai4i["UDI"].isin(merged["UDI"])
].copy()

# AI4I 원본 고장 플래그 → 한국어 라벨
fault_map = {
    "TWF": "공구 마모 고장",
    "HDF": "열 방출 장애",
    "PWF": "동력 고장",
    "OSF": "과부하 고장"
}

def get_label(row):
    if row["RNF"] != 0:
        return None

    active = [
        key for key in fault_map
        if row[key] == 1
    ]

    if len(active) == 1 and row["Machine failure"] == 1:
        return fault_map[active[0]]

    if len(active) == 0 and row["Machine failure"] == 0:
        return "정상"

    return None

# 라벨이 명확한 데이터만 사용
remaining["label"] = remaining.apply(get_label, axis=1)
remaining = remaining.dropna(subset=["label"])

# 고장 사례를 우선 추가하고 부족한 수는 정상 사례로 채움
need = 500 - len(base_train)

faults = remaining[remaining["label"] != "정상"]
normals = remaining[remaining["label"] == "정상"]

extra_faults = faults.sample(
    n=min(len(faults), need),
    random_state=42
)

extra_normals = normals.sample(
    n=need - len(extra_faults),
    random_state=42
)

extra = pd.concat([extra_faults, extra_normals])
train500 = pd.concat([base_train, extra], ignore_index=True)

print("확장 학습 데이터:", len(train500))
print(train500["label"].value_counts())

# 기존 변환 함수를 이용해 저장
import json

path = "/kaggle/working/train_500.jsonl"

with open(path, "w", encoding="utf-8") as f:
    for _, row in train500.iterrows():
        f.write(
            json.dumps(make_case(row), ensure_ascii=False) + "\n"
        )

print("저장 완료:", path)


확장 학습 데이터: 500
label
정상          314
열 방출 장애      66
동력 고장        48
과부하 고장       46
공구 마모 고장     26
Name: count, dtype: int64
저장 완료: /kaggle/working/train_500.jsonl


In [28]:

# 학습 데이터 중복 확인
print("중복 UDI:", train500["UDI"].duplicated().sum())

# 검증 및 시험 데이터와 겹치는지 확인
heldout = merged[
    merged["split"].isin(["validation", "test"])
]

overlap = set(train500["UDI"]) & set(heldout["UDI"])

print("검증/시험 데이터 중복:", len(overlap))

assert train500["UDI"].is_unique
assert len(overlap) == 0

print("데이터 검사 완료!")


중복 UDI: 0
검증/시험 데이터 중복: 0
데이터 검사 완료!


In [29]:


import json
import math

# 제품 유형 조회 (L, M, H)
type_map = ai4i.set_index("UDI")["Type"]

def make_case_v4(row):
    air = float(row["Air temperature [K]"])
    process = float(row["Process temperature [K]"])
    rpm = int(row["Rotational speed [rpm]"])
    torque = float(row["Torque [Nm]"])
    wear = int(row["Tool wear [min]"])

    state = {
        "product_type": str(type_map.loc[int(row["UDI"])]),
        "air_temperature_K": air,
        "process_temperature_K": process,
        "rotational_speed_rpm": rpm,
        "torque_Nm": torque,
        "tool_wear_min": wear,

        # 새로운 특징값
        "temperature_gap_K": round(process - air, 2),
        "power_W": round(
            torque * rpm * 2 * math.pi / 60, 2
        ),
        "wear_torque": round(wear * torque, 2)
    }

    return {
        "state": json.dumps(state, ensure_ascii=False),
        "questions": questions,
        "expected": {
            "diagnosis": row["label"]
        }
    }

# 기존 데이터 분할 유지
datasets = {
    "train": train500,
    "validation": merged[
        merged["split"] == "validation"
    ],
    "test": merged[
        merged["split"] == "test"
    ]
}

for name, subset in datasets.items():
    path = f"/kaggle/working/{name}_features.jsonl"

    with open(path, "w", encoding="utf-8") as f:
        for _, row in subset.iterrows():
            case = make_case_v4(row)
            f.write(
                json.dumps(case, ensure_ascii=False) + "\n"
            )

    print(name, len(subset), "완료")



train 500 완료
validation 80 완료
test 80 완료


In [31]:

!laya-train \
    --data /kaggle/working/train_features.jsonl \
    --base multilingual \
    --out /kaggle/working/laya_ai4i_v7 \
    --eval-data /kaggle/working/validation_features.jsonl \
    --epochs 16 \
    --micro-batch 4 \
    --grad-accum 4 \
    --loss rlcd \
    --seed 0


Reconstructing (incomplete total...): |            |  0.00B /  0.00B            

Fetching 5 files: 100%|█████████████████████████| 5/5 [00:00<00:00, 1233.40it/s]
Download complete:                                          |  0.00B            
Download complete:                                          |  0.00B            
Reconstruction complete: |                         |  0.00B /  0.00B            
eval items 80 from /kaggle/working/validation_features.jsonl (skipped {}, 0 overlapping training/calibration items)
train items 450, calibration items 50, eval items 80 (/kaggle/working/validation_features.jsonl), skipped {}, device cuda:0
optimizer budget: 450 train item(s), effective batch up to 16 (micro_batch 4 x grad_accum 4), 464 optimizer updates over 16 epoch(s)
/usr/local/lib/python3.13/dist-packages/laya/train.py:724: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.

In [32]:

import json
import laya
import pandas as pd
from sklearn.metrics import (
    classification_report,
    accuracy_score
)

model = laya.load("/kaggle/working/laya_ai4i_v7")

with open(
    "/kaggle/working/validation_features.jsonl",
    encoding="utf-8"
) as f:
    cases = [json.loads(line) for line in f]

y_true, y_pred = [], []

for case in cases:
    result = model.predict(
        case["state"],
        case["questions"]
    )

    y_true.append(case["expected"]["diagnosis"])
    y_pred.append(
        result["answers"]["diagnosis"]["choice"]
    )

print("v7 Accuracy:", accuracy_score(y_true, y_pred))
print("\n고장 유형별 성능:")
print(classification_report(
    y_true, y_pred, zero_division=0
))

print("\n오답 분석표:")
print(pd.crosstab(
    pd.Series(y_true, name="실제"),
    pd.Series(y_pred, name="예측")
))


v7 Accuracy: 0.9125

고장 유형별 성능:
              precision    recall  f1-score   support

    공구 마모 고장       0.80      0.50      0.62         8
      과부하 고장       1.00      0.94      0.97        16
       동력 고장       1.00      0.88      0.93        16
     열 방출 장애       0.95      1.00      0.98        20
          정상       0.80      1.00      0.89        20

    accuracy                           0.91        80
   macro avg       0.91      0.86      0.88        80
weighted avg       0.92      0.91      0.91        80


오답 분석표:
예측        공구 마모 고장  과부하 고장  동력 고장  열 방출 장애  정상
실제                                            
공구 마모 고장         4       0      0        0   4
과부하 고장           0      15      0        0   1
동력 고장            1       0     14        1   0
열 방출 장애          0       0      0       20   0
정상               0       0      0        0  20


In [33]:

import shutil
import os

model_dir = "/kaggle/working/laya_ai4i_v7"

assert os.path.isdir(model_dir), "v7 모델 폴더가 없습니다."

zip_path = shutil.make_archive(
    "/kaggle/working/laya_ai4i_v7_backup",
    "zip",
    root_dir="/kaggle/working",
    base_dir="laya_ai4i_v7"
)

print("v7 백업 완료:", zip_path)


v7 백업 완료: /kaggle/working/laya_ai4i_v7_backup.zip
